In [22]:
import numpy as np
from unified_planning.shortcuts import *
from unified_planning.engines import PlanGenerationResultStatus
import torch
import tqdm
import random

from unified_planning.shortcuts import Compiler, SequentialSimulator
from unified_planning.engines import CompilationKind
from unified_planning.plans import ActionInstance

In [23]:
def create_blocksworld_problem():
    # ---------------------------------------------------------
    # Types and objects
    # ---------------------------------------------------------

    Block = UserType("Block")

    A = Object("A", Block)
    B = Object("B", Block)
    C = Object("C", Block)

    problem = Problem("blocksworld_3_blocks")
    problem.add_objects([A, B, C])

    # ---------------------------------------------------------
    # Fluents
    # ---------------------------------------------------------

    # on(x, y): block x is directly on block y
    on = Fluent(
        "on",
        BoolType(),
        x=Block,
        y=Block,
    )

    # ontable(x): block x is on the table
    ontable = Fluent(
        "ontable",
        BoolType(),
        x=Block,
    )

    # clear(x): nothing is on top of x
    clear = Fluent(
        "clear",
        BoolType(),
        x=Block,
    )

    # holding(x): the hand is holding x
    holding = Fluent(
        "holding",
        BoolType(),
        x=Block,
    )

    # handempty: the single hand is empty
    handempty = Fluent(
        "handempty",
        BoolType(),
    )

    # Anything not explicitly true is false.
    problem.add_fluent(on, default_initial_value=False)
    problem.add_fluent(ontable, default_initial_value=False)
    problem.add_fluent(clear, default_initial_value=False)
    problem.add_fluent(holding, default_initial_value=False)
    problem.add_fluent(handempty, default_initial_value=False)

    # ---------------------------------------------------------
    # Action: pick_up(x)
    #
    # Pick a clear block up from the table.
    # ---------------------------------------------------------

    pick_up = InstantaneousAction(
        "pick_up",
        x=Block,
    )

    x = pick_up.parameter("x")

    pick_up.add_precondition(ontable(x))
    pick_up.add_precondition(clear(x))
    pick_up.add_precondition(handempty)

    pick_up.add_effect(ontable(x), False)
    pick_up.add_effect(clear(x), False)
    pick_up.add_effect(handempty, False)
    pick_up.add_effect(holding(x), True)

    problem.add_action(pick_up)

    # ---------------------------------------------------------
    # Action: put_down(x)
    #
    # Put a held block onto the table.
    # ---------------------------------------------------------

    put_down = InstantaneousAction(
        "put_down",
        x=Block,
    )

    x = put_down.parameter("x")

    put_down.add_precondition(holding(x))

    put_down.add_effect(holding(x), False)
    put_down.add_effect(ontable(x), True)
    put_down.add_effect(clear(x), True)
    put_down.add_effect(handempty, True)

    problem.add_action(put_down)

    # ---------------------------------------------------------
    # Action: stack(x, y)
    #
    # Put held block x on top of clear block y.
    # ---------------------------------------------------------

    stack = InstantaneousAction(
        "stack",
        x=Block,
        y=Block,
    )

    x = stack.parameter("x")
    y = stack.parameter("y")

    stack.add_precondition(holding(x))
    stack.add_precondition(clear(y))

    # Prevent nonsensical stack(A, A)
    stack.add_precondition(Not(Equals(x, y)))

    stack.add_effect(holding(x), False)
    stack.add_effect(clear(y), False)
    stack.add_effect(clear(x), True)
    stack.add_effect(handempty, True)
    stack.add_effect(on(x, y), True)

    problem.add_action(stack)

    # ---------------------------------------------------------
    # Action: unstack(x, y)
    #
    # Remove clear block x from block y.
    # ---------------------------------------------------------

    unstack = InstantaneousAction(
        "unstack",
        x=Block,
        y=Block,
    )

    x = unstack.parameter("x")
    y = unstack.parameter("y")

    unstack.add_precondition(on(x, y))
    unstack.add_precondition(clear(x))
    unstack.add_precondition(handempty)

    unstack.add_precondition(Not(Equals(x, y)))

    unstack.add_effect(on(x, y), False)
    unstack.add_effect(clear(y), True)
    unstack.add_effect(clear(x), False)
    unstack.add_effect(handempty, False)
    unstack.add_effect(holding(x), True)

    problem.add_action(unstack)

    # ---------------------------------------------------------
    # Initial state
    #
    #       A     B     C
    #      ---   ---   ---
    #           TABLE
    #
    # All three blocks start separately on the table.
    # ---------------------------------------------------------

    problem.set_initial_value(ontable(A), True)
    problem.set_initial_value(ontable(B), True)
    problem.set_initial_value(ontable(C), True)

    problem.set_initial_value(clear(A), True)
    problem.set_initial_value(clear(B), True)
    problem.set_initial_value(clear(C), True)

    problem.set_initial_value(handempty, True)

    # ---------------------------------------------------------
    # Goal
    #
    #       A
    #       B
    #       C
    #      ---
    #     TABLE
    #
    # A on B, B on C
    # ---------------------------------------------------------

    problem.add_goal(on(A, B))
    problem.add_goal(on(B, C))

    return problem


In [24]:
import random

from unified_planning.shortcuts import Compiler, SequentialSimulator
from unified_planning.engines import CompilationKind
from unified_planning.plans import ActionInstance


def _parameter_name(parameter):
    if parameter.is_object_exp():
        return parameter.object().name
    return str(parameter)


def _action_to_tuple(action_instance, max_arity):
    """
    stack(A, B) -> ["stack", "A", "B"]
    pick_up(A)  -> ["pick_up", "A", None]
    """
    result = [action_instance.action.name]

    result.extend(
        _parameter_name(p)
        for p in action_instance.actual_parameters
    )

    while len(result) < max_arity + 1:
        result.append(None)

    return result


def _pad_trajectory(trajectory, horizon, tuple_size):
    """
    If H is the maximum NUMBER OF ACTIONS, we use H+1 positions:

        actions..., END, PAD, PAD, ...

    Example, H=4:

        [pickup(A), stack(A,B)]

    becomes

        [
            pickup(A),
            stack(A,B),
            END,
            PAD,
            PAD
        ]

    Total sequence length = H + 1.
    """

    result = [list(action) for action in trajectory]

    end_token = ["END"] + [None] * (tuple_size - 1)
    pad_token = ["PAD"] + [None] * (tuple_size - 1)

    result.append(end_token)

    while len(result) < horizon + 1:
        result.append(list(pad_token))

    return result


def enumerate_trajectories_up_to_horizon(
    problem,
    horizon,
    k,
    seed=None,
    pad=True,
    include_empty_plan=False,
):
    """
    Enumerate ALL executable trajectories with lengths 1..H
    in a single DFS traversal.

    Returns:
        - every trajectory of length <= H whose final state
          satisfies the goal;
        - a uniform random sample of at most k of all other
          executable trajectories of length <= H.

    Therefore:

        len(result["trajectories"])
        =
        min(k, number of non-goal trajectories)

    A trajectory that reaches the goal at depth d is recorded
    as a plan, but search continues below it. Therefore longer
    executable sequences may also be plans if their final states
    satisfy the goal.
    """

    if horizon < 0:
        raise ValueError("horizon must be >= 0")

    if k < 0:
        raise ValueError("k must be >= 0")

    rng = random.Random(seed)

    max_arity = max(
        (len(action.parameters) for action in problem.actions),
        default=0,
    )

    tuple_size = max_arity + 1

    plans = []

    # Uniform reservoir sample of non-plan trajectories.
    sampled_trajectories = []
    num_non_plan_trajectories = 0

    # Useful statistics.
    num_executable_trajectories = 0

    # Optional counts by trajectory length.
    plans_by_length = {
        length: 0
        for length in range(horizon + 1)
    }

    trajectories_by_length = {
        length: 0
        for length in range(horizon + 1)
    }

    def store_plan(trajectory):
        if pad:
            plans.append(
                _pad_trajectory(
                    trajectory,
                    horizon,
                    tuple_size,
                )
            )
        else:
            plans.append(list(trajectory))

    def store_non_plan(trajectory):
        nonlocal num_non_plan_trajectories

        num_non_plan_trajectories += 1

        if k == 0:
            return

        if pad:
            candidate = _pad_trajectory(
                trajectory,
                horizon,
                tuple_size,
            )
        else:
            candidate = list(trajectory)

        # Initially fill the reservoir.
        if len(sampled_trajectories) < k:
            sampled_trajectories.append(candidate)
            return

        # Standard reservoir sampling.
        #
        # Every non-plan trajectory encountered so far has
        # equal probability of appearing in the final sample.
        j = rng.randrange(num_non_plan_trajectories)

        if j < k:
            sampled_trajectories[j] = candidate

    # ---------------------------------------------------------
    # Ground the problem once.
    # ---------------------------------------------------------

    with Compiler(
        problem_kind=problem.kind,
        compilation_kind=CompilationKind.GROUNDING,
    ) as grounder:

        grounding_result = grounder.compile(
            problem,
            CompilationKind.GROUNDING,
        )

        grounded_problem = grounding_result.problem
        grounded_actions = list(grounded_problem.actions)

        with SequentialSimulator(
            problem=grounded_problem
        ) as simulator:

            initial_state = simulator.get_initial_state()

            # -------------------------------------------------
            # Optional zero-length plan.
            # -------------------------------------------------

            if (
                include_empty_plan
                and simulator.is_goal(initial_state)
            ):
                store_plan([])
                plans_by_length[0] += 1

            # -------------------------------------------------
            # One DFS explores ALL horizons 1...H.
            # -------------------------------------------------

            def dfs(state, trajectory):
                nonlocal num_executable_trajectories

                depth = len(trajectory)

                # We have already classified this prefix.
                # Do not expand beyond H.
                if depth == horizon:
                    return

                for grounded_action in grounded_actions:

                    grounded_ai = ActionInstance(
                        grounded_action
                    )

                    if not simulator.is_applicable(
                        state,
                        grounded_ai,
                    ):
                        continue

                    next_state = simulator.apply(
                        state,
                        grounded_ai,
                    )

                    if next_state is None:
                        continue

                    original_ai = (
                        grounding_result
                        .map_back_action_instance(
                            grounded_ai
                        )
                    )

                    encoded_action = _action_to_tuple(
                        original_ai,
                        max_arity,
                    )

                    # Add one action to the current prefix.
                    trajectory.append(encoded_action)

                    new_depth = len(trajectory)

                    # -----------------------------------------
                    # THIS is the important change:
                    #
                    # classify every prefix, not only depth H.
                    # -----------------------------------------

                    num_executable_trajectories += 1

                    if simulator.is_goal(next_state):

                        store_plan(trajectory)

                        plans_by_length[new_depth] += 1

                    else:

                        store_non_plan(trajectory)

                        trajectories_by_length[new_depth] += 1

                    # -----------------------------------------
                    # Continue searching until H regardless of
                    # whether this prefix already reaches goal.
                    # -----------------------------------------

                    dfs(
                        next_state,
                        trajectory,
                    )

                    trajectory.pop()

            dfs(
                initial_state,
                [],
            )

    return {
        "plans": plans,
        "trajectories": sampled_trajectories,

        "num_plans": len(plans),

        "num_non_plan_trajectories":
            num_non_plan_trajectories,

        "num_executable_trajectories":
            num_executable_trajectories,

        "plans_by_length":
            plans_by_length,

        "trajectories_by_length":
            trajectories_by_length,

        "horizon": horizon,
    }

In [25]:
problem = create_blocksworld_problem()

data = enumerate_trajectories_up_to_horizon(
    problem=problem,
    horizon=6,
    k=1000,
    seed=42,
)

  *** Credits ***
  * In operation mode `Compiler` at line 180 of `/tmp/ipykernel_136253/3823043928.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.



In [26]:
print("Plans:")
print(data["plans_by_length"])

print("Non-plan trajectories:")
print(data["trajectories_by_length"])

Plans:
{0: 0, 1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 7}
Non-plan trajectories:
{0: 0, 1: 3, 2: 9, 3: 21, 4: 56, 5: 123, 6: 320}


In [27]:
for plan in data["plans"]:
    print(plan)

[['pick_up', 'A', None], ['put_down', 'A', None], ['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B'], ['END', None, None]]
[['pick_up', 'B', None], ['put_down', 'B', None], ['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B'], ['END', None, None]]
[['pick_up', 'B', None], ['stack', 'B', 'A'], ['unstack', 'B', 'A'], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B'], ['END', None, None]]
[['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['put_down', 'A', None], ['pick_up', 'A', None], ['stack', 'A', 'B'], ['END', None, None]]
[['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B'], ['END', None, None], ['PAD', None, None], ['PAD', None, None]]
[['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B'], ['unstack', 'A', 'B'], ['stack', 'A', 'B'], ['END', None, None]]
[['pick_up', 'B', None], ['stack', 'B', 'C'], ['uns

In [66]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Sequence

import torch


# ============================================================
# Special tokens
# ============================================================

PAD = "<PAD>"
MASK = "<MASK>"
NONE = "<NONE>"
END = "<END>"

INIT = "<INIT>"
GOAL = "<GOAL>"

SPECIAL_TOKENS = [
    PAD,
    MASK,
    NONE,
    END,
    INIT,
    GOAL,
]


# ============================================================
# Returned data structures
# ============================================================

@dataclass
class EncodedPlan:
    """
    Neural representation of one plan.

    tensor:
        Shape:
            [H + 1, tuple_width]

        H is the maximum number of REAL actions.

        The additional row is required for END.

    length:
        Number of real actions in the plan.
        Does not include END or PAD.
    """

    tensor: torch.LongTensor
    length: int


@dataclass
class EncodedContext:
    """
    Neural representation of the initial state and goal.

    tensor:
        Shape:
            [context_rows, tuple_width]

    attention_mask:
        Shape:
            [context_rows]

        True:
            meaningful row

        False:
            padding row

    num_initial_facts:
        Number of true facts in the initial state.

    num_goal_facts:
        Number of positive goal facts.
    """

    tensor: torch.LongTensor
    attention_mask: torch.BoolTensor

    num_initial_facts: int
    num_goal_facts: int


# ============================================================
# Plan / Planning Context Codec
# ============================================================

class PlanCodec:
    """
    Converts between symbolic classical-planning representations
    and categorical PyTorch tensors.

    The same vocabulary is used for:

        - action schemas
        - predicate schemas
        - objects
        - special tokens

    Example action:

        stack(A, B)

    becomes

        ["stack", "A", "B"]

    Example unary action:

        pick_up(A)

    becomes

        ["pick_up", "A", "<NONE>"]

    Example fluent:

        on(A, B)

    becomes

        ["on", "A", "B"]

    Example unary fluent:

        clear(A)

    becomes

        ["clear", "A", "<NONE>"]


    Horizon convention
    ------------------

    H = maximum number of REAL actions.

    Therefore:

        num_plan_slots = H + 1

    because every plan representation also requires an END row.

    For H = 4:

        [a1]
        [a2]
        [END]
        [PAD]
        [PAD]

    and a maximum-length plan is:

        [a1]
        [a2]
        [a3]
        [a4]
        [END]
    """

    # ========================================================
    # Construction
    # ========================================================

    def __init__(
        self,
        action_names: Sequence[str],
        predicate_names: Sequence[str],
        object_names: Sequence[str],
        max_action_arity: int,
        max_fluent_arity: int,
        horizon: int,
    ):

        if horizon < 0:
            raise ValueError(
                "horizon must be non-negative."
            )

        self.action_names = sorted(
            set(action_names)
        )

        self.predicate_names = sorted(
            set(predicate_names)
        )

        self.object_names = sorted(
            set(object_names)
        )

        self.max_action_arity = (
            max_action_arity
        )

        self.max_fluent_arity = (
            max_fluent_arity
        )

        # ----------------------------------------------------
        # One common tuple width is used everywhere.
        #
        # First entry:
        #     operator / predicate
        #
        # Remaining entries:
        #     arguments
        # ----------------------------------------------------

        self.max_arity = max(
            max_action_arity,
            max_fluent_arity,
        )

        self.tuple_width = (
            1 + self.max_arity
        )

        # ----------------------------------------------------
        # H = max REAL actions.
        #
        # Need an extra row for END.
        # ----------------------------------------------------

        self.horizon = horizon

        self.num_plan_slots = (
            horizon + 1
        )

        # ----------------------------------------------------
        # Construct one shared categorical vocabulary.
        # ----------------------------------------------------

        vocabulary = list(
            dict.fromkeys(
                SPECIAL_TOKENS
                + self.action_names
                + self.predicate_names
                + self.object_names
            )
        )

        self.token_to_id = {
            token: index
            for index, token
            in enumerate(vocabulary)
        }

        self.id_to_token = {
            index: token
            for token, index
            in self.token_to_id.items()
        }

        self.vocab_size = len(
            vocabulary
        )

        # ----------------------------------------------------
        # Convenient special-token IDs
        # ----------------------------------------------------

        self.pad_id = (
            self.token_to_id[PAD]
        )

        self.mask_id = (
            self.token_to_id[MASK]
        )

        self.none_id = (
            self.token_to_id[NONE]
        )

        self.end_id = (
            self.token_to_id[END]
        )

        self.init_id = (
            self.token_to_id[INIT]
        )

        self.goal_id = (
            self.token_to_id[GOAL]
        )

    # ========================================================
    # Construct directly from Unified Planning problem
    # ========================================================

    @classmethod
    def from_problem(
        cls,
        problem,
        horizon: int,
    ) -> "PlanCodec":

        action_names = [
            action.name
            for action in problem.actions
        ]

        predicate_names = [
            fluent.name
            for fluent in problem.fluents
        ]

        object_names = [
            obj.name
            for obj in problem.all_objects
        ]

        max_action_arity = max(
            (
                len(action.parameters)
                for action in problem.actions
            ),
            default=0,
        )

        max_fluent_arity = max(
            (
                len(fluent.signature)
                for fluent in problem.fluents
            ),
            default=0,
        )

        return cls(
            action_names=action_names,
            predicate_names=predicate_names,
            object_names=object_names,
            max_action_arity=max_action_arity,
            max_fluent_arity=max_fluent_arity,
            horizon=horizon,
        )

    # ========================================================
    # Basic categorical conversion
    # ========================================================

    def encode_token(
        self,
        token,
    ) -> int:
        """
        Convert a symbolic token to its categorical ID.

        Python None is interpreted as <NONE>.
        """

        if token is None:
            token = NONE

        token = str(token)

        if token not in self.token_to_id:
            raise ValueError(
                f"Unknown token {token!r}. "
                f"Known vocabulary: "
                f"{list(self.token_to_id.keys())}"
            )

        return self.token_to_id[token]

    def decode_token(
        self,
        token_id: int,
    ) -> str:
        """
        Convert categorical ID back into symbolic token.
        """

        token_id = int(token_id)

        if token_id not in self.id_to_token:
            raise ValueError(
                f"Unknown token ID: {token_id}"
            )

        return self.id_to_token[
            token_id
        ]

    # ========================================================
    # Standard special rows
    # ========================================================

    def end_row(self) -> list[str]:
        """
        [END, NONE, ..., NONE]
        """

        return (
            [END]
            + [NONE]
            * (self.tuple_width - 1)
        )

    def pad_row(self) -> list[str]:
        """
        [PAD, PAD, ..., PAD]
        """

        return (
            [PAD]
            * self.tuple_width
        )

    def mask_row(self) -> list[str]:
        """
        [MASK, MASK, ..., MASK]
        """

        return (
            [MASK]
            * self.tuple_width
        )

    def init_row(self) -> list[str]:
        """
        [INIT, NONE, ..., NONE]
        """

        return (
            [INIT]
            + [NONE]
            * (self.tuple_width - 1)
        )

    def goal_row(self) -> list[str]:
        """
        [GOAL, NONE, ..., NONE]
        """

        return (
            [GOAL]
            + [NONE]
            * (self.tuple_width - 1)
        )

    # ========================================================
    # Action representation
    # ========================================================

    def normalize_action(
        self,
        action: Sequence,
    ) -> list[str]:
        """
        Normalize an action to tuple_width entries.

        Examples:

            ["pick_up", "A"]
                ->
            ["pick_up", "A", "<NONE>"]

            ["pick_up", "A", None]
                ->
            ["pick_up", "A", "<NONE>"]

            ["stack", "A", "B"]
                ->
            ["stack", "A", "B"]
        """

        if len(action) == 0:
            raise ValueError(
                "Action cannot be empty."
            )

        if len(action) > self.tuple_width:
            raise ValueError(
                f"Action {action} contains "
                f"{len(action)} fields, but "
                f"tuple width is "
                f"{self.tuple_width}."
            )

        result = []

        for token in action:

            if token is None:
                result.append(NONE)

            else:
                result.append(
                    str(token)
                )

        while (
            len(result)
            < self.tuple_width
        ):
            result.append(NONE)

        return result

    def encode_action(
        self,
        action: Sequence,
    ) -> torch.LongTensor:
        """
        Encode one symbolic action tuple.
        """

        normalized = (
            self.normalize_action(action)
        )

        return torch.tensor(
            [
                self.encode_token(token)
                for token in normalized
            ],
            dtype=torch.long,
        )

    # ========================================================
    # Plan encoding
    # ========================================================

    def encode_plan(
        self,
        plan: Sequence[Sequence],
    ) -> EncodedPlan:
        """
        Encode a symbolic plan.

        Accepts either:

        1. Real actions only:

            [
                ["pick_up", "A", None],
                ["stack", "A", "B"],
            ]

        or

        2. A previously padded representation:

            [
                ["pick_up", "A", None],
                ["stack", "A", "B"],
                ["END", None, None],
                ["PAD", None, None],
                ...
            ]

        Both are converted to the canonical representation:

            actions...
            END
            PAD
            ...
            PAD

        with exactly H + 1 rows.
        """

        action_rows = []

        # ----------------------------------------------------
        # Extract only REAL actions from input.
        # ----------------------------------------------------

        for row_index, row in enumerate(
            plan
        ):

            if len(row) == 0:
                raise ValueError(
                    f"Empty row at "
                    f"index {row_index}."
                )

            operator = str(row[0])

            # Accept both older enumerator
            # representations and canonical tokens.
            if operator in (
                "END",
                END,
            ):
                break

            if operator in (
                "PAD",
                PAD,
            ):
                break

            action_rows.append(
                self.normalize_action(row)
            )

        # ----------------------------------------------------
        # Horizon applies ONLY to real actions.
        # ----------------------------------------------------

        action_length = len(
            action_rows
        )

        if (
            action_length
            > self.horizon
        ):
            raise ValueError(
                f"Plan contains "
                f"{action_length} real actions, "
                f"but horizon H="
                f"{self.horizon} allows at most "
                f"{self.horizon} actions."
            )

        # ----------------------------------------------------
        # Canonical form:
        #
        # action...
        # END
        # PAD...
        # ----------------------------------------------------

        rows = list(
            action_rows
        )

        rows.append(
            self.end_row()
        )

        while (
            len(rows)
            < self.num_plan_slots
        ):
            rows.append(
                self.pad_row()
            )

        if (
            len(rows)
            != self.num_plan_slots
        ):
            raise RuntimeError(
                "Internal codec error: "
                f"expected "
                f"{self.num_plan_slots} rows, "
                f"got {len(rows)}."
            )

        tensor = torch.tensor(
            [
                [
                    self.encode_token(
                        token
                    )
                    for token in row
                ]
                for row in rows
            ],
            dtype=torch.long,
        )

        return EncodedPlan(
            tensor=tensor,
            length=action_length,
        )

    # ========================================================
    # Plan decoding
    # ========================================================

    def decode_plan(
        self,
        tensor: torch.Tensor,
    ) -> list[list[str | None]]:
        """
        Decode a neural tensor into REAL ACTIONS only.

        END and PAD are representation-level tokens and are
        therefore not returned.

        Desired round-trip property:

            decode_plan(
                encode_plan(plan).tensor
            )

        returns the original real action sequence.
        """

        tensor = (
            tensor
            .detach()
            .cpu()
        )

        # ----------------------------------------------------
        # Accept flattened tensors too.
        # ----------------------------------------------------

        if tensor.ndim == 1:

            expected_size = (
                self.num_plan_slots
                * self.tuple_width
            )

            if (
                tensor.numel()
                != expected_size
            ):
                raise ValueError(
                    f"Expected "
                    f"{expected_size} categorical "
                    f"entries, got "
                    f"{tensor.numel()}."
                )

            tensor = tensor.reshape(
                self.num_plan_slots,
                self.tuple_width,
            )

        expected_shape = (
            self.num_plan_slots,
            self.tuple_width,
        )

        if (
            tuple(tensor.shape)
            != expected_shape
        ):
            raise ValueError(
                f"Expected tensor shape "
                f"{expected_shape}, got "
                f"{tuple(tensor.shape)}."
            )

        plan = []

        for row_index, row in enumerate(
            tensor
        ):

            tokens = [
                self.decode_token(
                    token.item()
                )
                for token in row
            ]

            operator = tokens[0]

            # ------------------------------------------------
            # END terminates real plan.
            # ------------------------------------------------

            if operator == END:
                break

            # ------------------------------------------------
            # PAD before END is malformed, but this function
            # is a decoder rather than validator.
            # ------------------------------------------------

            if operator == PAD:
                break

            if operator == MASK:
                raise ValueError(
                    "Cannot decode completed "
                    f"plan: MASK remains "
                    f"at row {row_index}."
                )

            action = [
                operator
            ]

            for token in tokens[1:]:

                if token == NONE:
                    action.append(None)

                elif token in (
                    PAD,
                    END,
                    MASK,
                    INIT,
                    GOAL,
                ):
                    raise ValueError(
                        f"Invalid token {token} "
                        f"inside action at row "
                        f"{row_index}: {tokens}"
                    )

                else:
                    action.append(
                        token
                    )

            plan.append(
                action
            )

        return plan

    # ========================================================
    # DFM all-mask source plan
    # ========================================================

    def masked_plan(
        self,
    ) -> torch.LongTensor:
        """
        Return the DFM source plan X_0.

        Shape:

            [H + 1, tuple_width]

        Every categorical variable initially contains MASK.
        """

        return torch.full(
            (
                self.num_plan_slots,
                self.tuple_width,
            ),
            fill_value=self.mask_id,
            dtype=torch.long,
        )

    # ========================================================
    # Fluent representation
    # ========================================================

    def fluent_to_tuple(
        self,
        fluent_exp,
    ) -> list[str]:
        """
        Convert a grounded Unified Planning fluent expression
        to the shared symbolic tuple representation.

        Examples:

            on(A, B)
                ->
            ["on", "A", "B"]

            clear(A)
                ->
            ["clear", "A", "<NONE>"]

            handempty
                ->
            ["handempty", "<NONE>", "<NONE>"]
        """

        if (
            not fluent_exp
            .is_fluent_exp()
        ):
            raise ValueError(
                "Expected grounded fluent "
                f"expression, got "
                f"{fluent_exp}."
            )

        result = [
            fluent_exp
            .fluent()
            .name
        ]

        for arg in fluent_exp.args:

            if not arg.is_object_exp():
                raise ValueError(
                    "Only grounded object "
                    "arguments are currently "
                    "supported. "
                    f"Got {arg} in "
                    f"{fluent_exp}."
                )

            result.append(
                arg.object().name
            )

        while (
            len(result)
            < self.tuple_width
        ):
            result.append(NONE)

        if (
            len(result)
            > self.tuple_width
        ):
            raise ValueError(
                f"Fluent {fluent_exp} "
                "exceeds codec tuple width."
            )

        return result

    # ========================================================
    # Goal extraction
    # ========================================================

    def _extract_positive_goal_atoms(
        self,
        expr,
    ):
        """
        Extract positive grounded fluent atoms.

        Supported:

            on(A, B)

        and:

            And(
                on(A, B),
                on(B, C)
            )

        Negative, disjunctive, numeric, etc. goals are
        deliberately rejected in the current Blocks World
        version.
        """

        if expr.is_fluent_exp():
            return [expr]

        if expr.is_and():

            atoms = []

            for arg in expr.args:
                atoms.extend(
                    self
                    ._extract_positive_goal_atoms(
                        arg
                    )
                )

            return atoms

        raise ValueError(
            "Current context encoding "
            "supports only positive "
            "Boolean fluent goals. "
            f"Unsupported goal: {expr}"
        )

    # ========================================================
    # Initial state + goal context encoding
    # ========================================================

    def encode_problem_context(
        self,
        problem,
    ) -> EncodedContext:
        """
        Encode the problem's CURRENT initial state and goal.

        Representation:

            [INIT, NONE, NONE]

            initial fact
            initial fact
            ...
            PAD
            PAD

            [GOAL, NONE, NONE]

            goal fact
            goal fact
            ...
            PAD
            PAD


        Each section receives one slot for every grounded
        fluent in the problem.

        If n is the number of grounded fluents:

            context_rows = 2 + 2n


        Only TRUE initial-state fluents are serialized.

        For the positive STRIPS goal representation used here,
        omitted goal fluents mean "not required", NOT false.
        """

        # ----------------------------------------------------
        # Read initial values once.
        # ----------------------------------------------------

        initial_values = (
            problem.initial_values
        )

        num_ground_fluents = len(
            initial_values
        )

        # ----------------------------------------------------
        # TRUE initial facts
        # ----------------------------------------------------

        initial_facts = []

        for (
            fluent_exp,
            value,
        ) in initial_values.items():

            # Current project assumes Boolean STRIPS fluents.
            if value.is_true():

                initial_facts.append(
                    self.fluent_to_tuple(
                        fluent_exp
                    )
                )

        # ----------------------------------------------------
        # Canonical ordering.
        #
        # Avoid multiple sequence encodings for same logical
        # state.
        # ----------------------------------------------------

        initial_facts.sort(
            key=lambda row:
            tuple(row)
        )

        # ----------------------------------------------------
        # Goal atoms
        # ----------------------------------------------------

        goal_atoms = []

        for goal in problem.goals:

            goal_atoms.extend(
                self
                ._extract_positive_goal_atoms(
                    goal
                )
            )

        goal_facts = [
            self.fluent_to_tuple(
                goal_atom
            )
            for goal_atom
            in goal_atoms
        ]

        goal_facts.sort(
            key=lambda row:
            tuple(row)
        )

        # ----------------------------------------------------
        # Capacity checks
        # ----------------------------------------------------

        if (
            len(initial_facts)
            > num_ground_fluents
        ):
            raise RuntimeError(
                "Initial-state fact count "
                "exceeds number of grounded "
                "fluents."
            )

        if (
            len(goal_facts)
            > num_ground_fluents
        ):
            raise ValueError(
                "Goal fact count exceeds "
                "available grounded-fluent "
                "capacity."
            )

        rows = []
        mask = []

        # ====================================================
        # INITIAL STATE SECTION
        # ====================================================

        rows.append(
            self.init_row()
        )

        mask.append(True)

        for fact in initial_facts:

            rows.append(
                fact
            )

            mask.append(True)

        # Pad state section to exactly n fluent positions.

        for _ in range(
            num_ground_fluents
            - len(initial_facts)
        ):

            rows.append(
                self.pad_row()
            )

            mask.append(False)

        # ====================================================
        # GOAL SECTION
        # ====================================================

        rows.append(
            self.goal_row()
        )

        mask.append(True)

        for fact in goal_facts:

            rows.append(
                fact
            )

            mask.append(True)

        # Pad goal section.

        for _ in range(
            num_ground_fluents
            - len(goal_facts)
        ):

            rows.append(
                self.pad_row()
            )

            mask.append(False)

        # ----------------------------------------------------
        # Convert symbolic rows into categorical IDs.
        # ----------------------------------------------------

        tensor = torch.tensor(
            [
                [
                    self.encode_token(
                        token
                    )
                    for token in row
                ]
                for row in rows
            ],
            dtype=torch.long,
        )

        attention_mask = (
            torch.tensor(
                mask,
                dtype=torch.bool,
            )
        )

        return EncodedContext(
            tensor=tensor,
            attention_mask=attention_mask,
            num_initial_facts=len(
                initial_facts
            ),
            num_goal_facts=len(
                goal_facts
            ),
        )

    # ========================================================
    # Combine context and plan
    # ========================================================

    def combine_context_and_plan(
        self,
        context: EncodedContext,
        plan_tensor: torch.Tensor,
    ) -> tuple[
        torch.LongTensor,
        torch.BoolTensor,
    ]:
        """
        Combine context and plan into one structured tensor.

        Output:

            [
                context
                ----
                plan
            ]

        The context padding mask is preserved.

        Every plan row is considered active because even PAD/END
        are categorical variables the DFM may need to predict.
        """

        plan_tensor = (
            plan_tensor
            .detach()
            .cpu()
        )

        expected_plan_shape = (
            self.num_plan_slots,
            self.tuple_width,
        )

        if (
            tuple(plan_tensor.shape)
            != expected_plan_shape
        ):
            raise ValueError(
                f"Expected plan tensor shape "
                f"{expected_plan_shape}, got "
                f"{tuple(plan_tensor.shape)}."
            )

        combined = torch.cat(
            [
                context.tensor,
                plan_tensor,
            ],
            dim=0,
        )

        plan_mask = torch.ones(
            self.num_plan_slots,
            dtype=torch.bool,
        )

        combined_mask = torch.cat(
            [
                context.attention_mask,
                plan_mask,
            ],
            dim=0,
        )

        return (
            combined,
            combined_mask,
        )

    # ========================================================
    # Flatten / unflatten PLAN tensor
    # ========================================================

    def flatten_plan(
        self,
        tensor: torch.Tensor,
    ) -> torch.LongTensor:
        """
        [H + 1, tuple_width]
            ->
        [(H + 1) * tuple_width]
        """

        return tensor.reshape(-1)

    def unflatten_plan(
        self,
        tensor: torch.Tensor,
    ) -> torch.LongTensor:
        """
        [(H + 1) * tuple_width]
            ->
        [H + 1, tuple_width]
        """

        expected_size = (
            self.num_plan_slots
            * self.tuple_width
        )

        if (
            tensor.numel()
            != expected_size
        ):
            raise ValueError(
                f"Expected {expected_size} "
                f"categorical entries, got "
                f"{tensor.numel()}."
            )

        return tensor.reshape(
            self.num_plan_slots,
            self.tuple_width,
        )

    # ========================================================
    # General flattening helper
    # ========================================================

    @staticmethod
    def flatten(
        tensor: torch.Tensor,
    ) -> torch.LongTensor:
        """
        Flatten any structured categorical tensor.
        """

        return tensor.reshape(-1)

    # ========================================================
    # Vocabulary debugging
    # ========================================================

    def print_vocabulary(
        self,
    ):
        """
        Print categorical vocabulary in ID order.
        """

        for index in range(
            self.vocab_size
        ):

            print(
                f"{index:3d}: "
                f"{self.id_to_token[index]}"
            )

In [67]:
problem = create_blocksworld_problem()

H = 6

codec = PlanCodec.from_problem(
    problem,
    horizon=H,
)

codec.print_vocabulary()

  0: <PAD>
  1: <MASK>
  2: <NONE>
  3: <END>
  4: <INIT>
  5: <GOAL>
  6: pick_up
  7: put_down
  8: stack
  9: unstack
 10: clear
 11: handempty
 12: holding
 13: on
 14: ontable
 15: A
 16: B
 17: C


In [68]:
plan = [
    ["pick_up", "B", None],
    ["stack", "B", "C"],
    ["pick_up", "A", None],
    ["stack", "A", "B"],
]

In [69]:
encoded = codec.encode_plan(plan)

print(encoded.tensor)
print(encoded.tensor.shape)
print(encoded.length)

tensor([[ 6, 16,  2],
        [ 8, 16, 17],
        [ 6, 15,  2],
        [ 8, 15, 16],
        [ 3,  2,  2],
        [ 0,  0,  0],
        [ 0,  0,  0]])
torch.Size([7, 3])
4


In [70]:
context = (
    codec.encode_problem_context(
        problem
    )
)

print(context.tensor.shape)
print(context.attention_mask.shape)

torch.Size([40, 3])
torch.Size([40])


In [50]:
decoded = codec.decode_plan(encoded.tensor)

print(decoded)

[['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B']]


In [51]:
original = [
    ["pick_up", "B", None],
    ["stack", "B", "C"],
    ["pick_up", "A", None],
    ["stack", "A", "B"],
]

encoded = codec.encode_plan(original)

decoded = codec.decode_plan(encoded.tensor)

reencoded = codec.encode_plan(decoded)

print(decoded)
print(torch.equal(
    encoded.tensor,
    reencoded.tensor,
))

[['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B']]
True


In [54]:
def encode_enumeration_result(
    result,
    codec: PlanCodec,
):
    """
    Convert an enumeration result into PyTorch tensors.

    Assumes the enumeration function already added END/PAD.
    """

    encoded_plans = [
        codec.encode_plan(
            p,
            # already_padded=True,
        ).tensor
        for p in result["plans"]
    ]

    encoded_trajectories = [
        codec.encode_plan(
            p,
            # already_padded=True,
        ).tensor
        for p in result["trajectories"]
    ]

    if encoded_plans:
        plans_tensor = torch.stack(
            encoded_plans
        )
    else:
        plans_tensor = torch.empty(
            (
                0,
                codec.num_plan_slots,
                codec.tuple_width,
            ),
            dtype=torch.long,
        )

    if encoded_trajectories:
        trajectories_tensor = torch.stack(
            encoded_trajectories
        )
    else:
        trajectories_tensor = torch.empty(
            (
                0,
                codec.num_plan_slots,
                codec.tuple_width,
            ),
            dtype=torch.long,
        )

    return {
        "plans": plans_tensor,
        "trajectories": trajectories_tensor,
    }

In [55]:
raw_data = enumerate_trajectories_up_to_horizon(
    problem=problem,
    horizon=6,
    k=1000,
    seed=42,
)

codec = PlanCodec.from_problem(
    problem,
    horizon=6,
)

data = encode_enumeration_result(
    raw_data,
    codec,
)

print(data["plans"].shape)
print(data["trajectories"].shape)

  *** Credits ***
  * In operation mode `Compiler` at line 180 of `/tmp/ipykernel_136253/3823043928.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.

torch.Size([8, 7, 3])
torch.Size([532, 7, 3])


In [56]:
x0 = codec.masked_plan()

In [57]:
x0

tensor([[1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]])

In [61]:
from dataclasses import dataclass
from typing import Optional

import torch

from unified_planning.shortcuts import SequentialSimulator


@dataclass
class PlanValidationResult:
    """
    Result of validating a neural-network-generated plan.
    """

    valid: bool

    # True iff all actions were executable AND the final state
    # satisfies the planning goal.
    solves_problem: bool

    # Decoded symbolic action sequence.
    plan: list

    # Number of successfully executed actions.
    executed_actions: int

    # Index of the action/row where validation failed.
    # None if there was no execution/syntax failure.
    failure_step: Optional[int]

    # Human-readable explanation.
    reason: str

    # Final reachable state, when available.
    final_state: object = None

def _normalize_plan_tensor(
    plan_tensor: torch.Tensor,
    codec,
) -> torch.Tensor:
    """
    Normalize a generated plan tensor.

    Convention:
        H = maximum number of REAL actions
        number of neural plan slots = H + 1

    The additional slot is reserved for END.

    Expected structured shape:
        [H + 1, tuple_width]

    Expected flattened shape:
        [(H + 1) * tuple_width]
    """

    tensor = plan_tensor.detach().cpu()

    expected_rows = codec.horizon + 1
    expected_cols = codec.tuple_width

    # ---------------------------------------------
    # Flattened representation
    # ---------------------------------------------
    if tensor.ndim == 1:

        expected_size = (
            expected_rows
            * expected_cols
        )

        if tensor.numel() != expected_size:
            raise ValueError(
                f"Wrong flattened tensor size. "
                f"Horizon H={codec.horizon} allows "
                f"{codec.horizon} actions plus one END slot, "
                f"so expected {expected_size} categorical "
                f"entries, but got {tensor.numel()}."
            )

        tensor = tensor.reshape(
            expected_rows,
            expected_cols,
        )

    # ---------------------------------------------
    # Structured representation
    # ---------------------------------------------
    elif tensor.ndim == 2:

        expected_shape = (
            expected_rows,
            expected_cols,
        )

        if tuple(tensor.shape) != expected_shape:
            raise ValueError(
                f"Wrong tensor shape. "
                f"Horizon H={codec.horizon} means "
                f"H+1={expected_rows} plan slots "
                f"(actions + END). "
                f"Expected {expected_shape}, "
                f"got {tuple(tensor.shape)}."
            )

    else:
        raise ValueError(
            "Plan tensor must be either 1D "
            "(flattened) or 2D."
        )

    return tensor

def validate_tensor_plan(
    problem,
    plan_tensor: torch.Tensor,
    codec,
    strict_format: bool = True,
) -> PlanValidationResult:
    """
    Validate a generated plan tensor against a Unified Planning problem.

    Checks:

        1. Tensor shape.
        2. No MASK tokens remain.
        3. END / PAD structure is valid.
        4. Action schema exists.
        5. Correct number of arguments.
        6. Referenced objects exist.
        7. Every action is applicable when executed.
        8. Final state satisfies the goal.

    Parameters
    ----------
    problem:
        Unified Planning Problem.

    plan_tensor:
        Tensor containing categorical token IDs.

        Expected structured shape:

            [H + 1, tuple_width]

        A flattened tensor is also accepted.

    codec:
        The PlanCodec used to encode/decode the plan.

    strict_format:
        If True, enforce the canonical representation:

            ACTION
            ACTION
            ...
            END
            PAD
            PAD
            ...

        This is recommended for evaluating DFM outputs.

    Returns
    -------
    PlanValidationResult
    """

    # ---------------------------------------------------------
    # 1. Normalize tensor shape
    # ---------------------------------------------------------

    try:
        tensor = _normalize_plan_tensor(
        plan_tensor,
        codec,
    )   

    except ValueError as e:
        return PlanValidationResult(
            valid=False,
            solves_problem=False,
            plan=[],
            executed_actions=0,
            failure_step=None,
            reason=str(e),
        )

    # ---------------------------------------------------------
    # 2. Decode tensor rows into tokens
    # ---------------------------------------------------------

    rows = []

    try:
        for row in tensor:
            decoded_row = [
                codec.decode_token(token.item())
                for token in row
            ]
            rows.append(decoded_row)

    except ValueError as e:
        return PlanValidationResult(
            valid=False,
            solves_problem=False,
            plan=[],
            executed_actions=0,
            failure_step=None,
            reason=f"Unknown categorical token: {e}",
        )

    # ---------------------------------------------------------
    # Useful lookup tables
    # ---------------------------------------------------------

    actions_by_name = {
        action.name: action
        for action in problem.actions
    }

    objects_by_name = {
        obj.name: obj
        for obj in problem.all_objects
    }

    # ---------------------------------------------------------
    # 3. Parse the generated plan
    # ---------------------------------------------------------

    decoded_plan = []

    seen_end = False

    for row_index, row in enumerate(rows):

        operator = row[0]
        arguments = row[1:]

        # -----------------------------------------------------
        # MASK is never allowed in a completed generation.
        # -----------------------------------------------------

        if "<MASK>" in row:
            return PlanValidationResult(
                valid=False,
                solves_problem=False,
                plan=decoded_plan,
                executed_actions=0,
                failure_step=row_index,
                reason=(
                    f"MASK token remains in row "
                    f"{row_index}: {row}"
                ),
            )

        # -----------------------------------------------------
        # END
        # -----------------------------------------------------

        if operator == "<END>":

            if strict_format:
                expected_arguments = [
                    "<NONE>"
                ] * codec.max_action_arity

                if arguments != expected_arguments:
                    return PlanValidationResult(
                        valid=False,
                        solves_problem=False,
                        plan=decoded_plan,
                        executed_actions=0,
                        failure_step=row_index,
                        reason=(
                            "Malformed END row: "
                            f"{row}"
                        ),
                    )

            seen_end = True
            continue

        # -----------------------------------------------------
        # PAD
        # -----------------------------------------------------

        if operator == "<PAD>":

            if strict_format:

                if not seen_end:
                    return PlanValidationResult(
                        valid=False,
                        solves_problem=False,
                        plan=decoded_plan,
                        executed_actions=0,
                        failure_step=row_index,
                        reason=(
                            "PAD encountered before END "
                            f"at row {row_index}."
                        ),
                    )

                if any(
                    token != "<PAD>"
                    for token in row
                ):
                    return PlanValidationResult(
                        valid=False,
                        solves_problem=False,
                        plan=decoded_plan,
                        executed_actions=0,
                        failure_step=row_index,
                        reason=(
                            "Malformed PAD row: "
                            f"{row}"
                        ),
                    )

            continue

        # -----------------------------------------------------
        # No action may occur after END.
        # -----------------------------------------------------

        if seen_end:

            return PlanValidationResult(
                valid=False,
                solves_problem=False,
                plan=decoded_plan,
                executed_actions=0,
                failure_step=row_index,
                reason=(
                    "Action encountered after END: "
                    f"{row}"
                ),
            )

        # -----------------------------------------------------
        # Operator must correspond to a real action schema.
        # -----------------------------------------------------

        if operator not in actions_by_name:

            return PlanValidationResult(
                valid=False,
                solves_problem=False,
                plan=decoded_plan,
                executed_actions=0,
                failure_step=row_index,
                reason=(
                    f"Unknown action schema "
                    f"{operator!r}."
                ),
            )

        action = actions_by_name[operator]
        arity = len(action.parameters)

        # -----------------------------------------------------
        # Required arguments
        # -----------------------------------------------------

        real_arguments = arguments[:arity]

        for arg_index, argument in enumerate(
            real_arguments
        ):
            if argument in (
                "<NONE>",
                "<PAD>",
                "<END>",
                "<MASK>",
            ):
                return PlanValidationResult(
                    valid=False,
                    solves_problem=False,
                    plan=decoded_plan,
                    executed_actions=0,
                    failure_step=row_index,
                    reason=(
                        f"Missing/invalid argument "
                        f"{arg_index} of action "
                        f"{operator}: {row}"
                    ),
                )

            if argument not in objects_by_name:
                return PlanValidationResult(
                    valid=False,
                    solves_problem=False,
                    plan=decoded_plan,
                    executed_actions=0,
                    failure_step=row_index,
                    reason=(
                        f"Unknown object "
                        f"{argument!r} in action "
                        f"{operator}."
                    ),
                )

        # -----------------------------------------------------
        # Unused argument positions must be NONE.
        #
        # pickup(A) should be
        #
        #     [pickup, A, NONE]
        #
        # not
        #
        #     [pickup, A, B]
        # -----------------------------------------------------

        unused_arguments = arguments[arity:]

        if strict_format and any(
            arg != "<NONE>"
            for arg in unused_arguments
        ):
            return PlanValidationResult(
                valid=False,
                solves_problem=False,
                plan=decoded_plan,
                executed_actions=0,
                failure_step=row_index,
                reason=(
                    f"Action {operator} has unexpected "
                    f"arguments: {row}"
                ),
            )

        decoded_plan.append(
            [operator] + real_arguments
        )

    # ---------------------------------------------------------
    # In strict mode, require an explicit END.
    # ---------------------------------------------------------

    if strict_format and not seen_end:

        return PlanValidationResult(
            valid=False,
            solves_problem=False,
            plan=decoded_plan,
            executed_actions=0,
            failure_step=None,
            reason="Generated plan contains no END token.",
        )

    # ---------------------------------------------------------
    # 4. Execute the decoded plan using UP
    # ---------------------------------------------------------

    with SequentialSimulator(
        problem=problem
    ) as simulator:

        state = simulator.get_initial_state()

        # Empty plan may itself solve the problem.
        if len(decoded_plan) == 0:

            goal_reached = simulator.is_goal(state)

            return PlanValidationResult(
                valid=True,
                solves_problem=goal_reached,
                plan=[],
                executed_actions=0,
                failure_step=None,
                reason=(
                    "Empty plan satisfies the goal."
                    if goal_reached
                    else
                    "Empty plan is executable but "
                    "does not satisfy the goal."
                ),
                final_state=state,
            )

        # -----------------------------------------------------
        # Sequential execution
        # -----------------------------------------------------

        for step, symbolic_action in enumerate(
            decoded_plan
        ):

            action_name = symbolic_action[0]
            argument_names = symbolic_action[1:]

            action = actions_by_name[action_name]

            objects = [
                objects_by_name[name]
                for name in argument_names
            ]

            # Calling a UP action with objects constructs an
            # ActionInstance:
            #
            #     stack(A, B)
            #
            try:
                action_instance = action(*objects)

            except Exception as e:
                return PlanValidationResult(
                    valid=False,
                    solves_problem=False,
                    plan=decoded_plan,
                    executed_actions=step,
                    failure_step=step,
                    reason=(
                        f"Could not instantiate "
                        f"{symbolic_action}: {e}"
                    ),
                    final_state=state,
                )

            # -------------------------------------------------
            # Check applicability BEFORE applying.
            # -------------------------------------------------

            try:
                applicable = simulator.is_applicable(
                    state,
                    action_instance,
                )

            except Exception as e:
                return PlanValidationResult(
                    valid=False,
                    solves_problem=False,
                    plan=decoded_plan,
                    executed_actions=step,
                    failure_step=step,
                    reason=(
                        f"Error checking applicability of "
                        f"{symbolic_action}: {e}"
                    ),
                    final_state=state,
                )

            if not applicable:

                return PlanValidationResult(
                    valid=False,
                    solves_problem=False,
                    plan=decoded_plan,
                    executed_actions=step,
                    failure_step=step,
                    reason=(
                        f"Action {symbolic_action} is "
                        f"not applicable at step {step}."
                    ),
                    final_state=state,
                )

            # -------------------------------------------------
            # Apply transition
            # -------------------------------------------------

            next_state = simulator.apply(
                state,
                action_instance,
            )

            if next_state is None:

                return PlanValidationResult(
                    valid=False,
                    solves_problem=False,
                    plan=decoded_plan,
                    executed_actions=step,
                    failure_step=step,
                    reason=(
                        f"Failed to apply action "
                        f"{symbolic_action} at step {step}."
                    ),
                    final_state=state,
                )

            state = next_state

        # -----------------------------------------------------
        # 5. Goal test
        # -----------------------------------------------------

        goal_reached = simulator.is_goal(state)

        if goal_reached:

            return PlanValidationResult(
                valid=True,
                solves_problem=True,
                plan=decoded_plan,
                executed_actions=len(decoded_plan),
                failure_step=None,
                reason="Plan is valid and satisfies the goal.",
                final_state=state,
            )

        return PlanValidationResult(
            valid=True,
            solves_problem=False,
            plan=decoded_plan,
            executed_actions=len(decoded_plan),
            failure_step=None,
            reason=(
                "All actions are executable, but the "
                "final state does not satisfy the goal."
            ),
            final_state=state,
        )

In [62]:
problem = create_blocksworld_problem()

codec = PlanCodec.from_problem(
    problem,
    horizon=6,
)

plan = [['pick_up', 'A', None], ['put_down', 'A', None], ['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B'], ['END', None, None]]

encoded = codec.encode_plan(plan)

result = validate_tensor_plan(
    problem,
    encoded.tensor,
    codec,
)

print(result)

PlanValidationResult(valid=True, solves_problem=True, plan=[['pick_up', 'A'], ['put_down', 'A'], ['pick_up', 'B'], ['stack', 'B', 'C'], ['pick_up', 'A'], ['stack', 'A', 'B']], executed_actions=6, failure_step=None, reason='Plan is valid and satisfies the goal.', final_state={clear(A): true, handempty: true, on(A, B): true, on(B, C): true, ontable(C): true})


In [ ]:
# dfm.py

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ============================================================
# Configuration
# ============================================================

@dataclass
class DFMConfig:
    vocab_size: int

    tuple_width: int

    # Number of rows in encoded initial-state + goal context.
    context_rows: int

    # H + 1, because END occupies one extra plan row.
    plan_slots: int

    d_model: int = 256
    nhead: int = 8
    num_layers: int = 6
    dim_feedforward: int = 1024
    dropout: float = 0.1


# ============================================================
# Time embedding
# ============================================================

def sinusoidal_time_embedding(
    t: torch.Tensor,
    dim: int,
) -> torch.Tensor:
    """
    Convert t in [0,1] to a sinusoidal embedding.

    Parameters
    ----------
    t:
        Shape [batch]

    Returns
    -------
    Shape [batch, dim]
    """

    if t.ndim != 1:
        raise ValueError(
            f"Expected t shape [B], got {t.shape}"
        )

    half = dim // 2

    device = t.device

    if half == 0:
        return t[:, None]

    frequencies = torch.exp(
        torch.linspace(
            0.0,
            math.log(10000.0),
            half,
            device=device,
        )
    )

    angles = (
        2.0
        * math.pi
        * t[:, None]
        * frequencies[None, :]
    )

    embedding = torch.cat(
        [
            torch.sin(angles),
            torch.cos(angles),
        ],
        dim=-1,
    )

    if embedding.shape[-1] < dim:
        embedding = F.pad(
            embedding,
            (0, dim - embedding.shape[-1]),
        )

    return embedding


# ============================================================
# Planning-specific output vocabulary mask
# ============================================================

def build_plan_output_mask(
    codec,
) -> torch.BoolTensor:
    """
    Static structural mask for network outputs.

    Shape:

        [plan_slots, tuple_width, vocab_size]

    Operator position may generate:

        action names
        END
        PAD

    Argument positions may generate:

        object names
        NONE
        PAD

    MASK is deliberately NOT a valid clean target.

    This does not enforce action-schema-dependent constraints such
    as:

        pick_up(A, NONE)

    versus

        pick_up(A, B)

    The validator will catch those. They can be added later as
    dynamic masks.
    """

    mask = torch.zeros(
        (
            codec.num_plan_slots,
            codec.tuple_width,
            codec.vocab_size,
        ),
        dtype=torch.bool,
    )

    # --------------------------------------------------------
    # Operator position
    # --------------------------------------------------------

    operator_ids = [
        codec.token_to_id[name]
        for name in codec.action_names
    ]

    operator_ids += [
        codec.end_id,
        codec.pad_id,
    ]

    mask[:, 0, operator_ids] = True

    # --------------------------------------------------------
    # Argument positions
    # --------------------------------------------------------

    argument_ids = [
        codec.token_to_id[name]
        for name in codec.object_names
    ]

    argument_ids += [
        codec.none_id,
        codec.pad_id,
    ]

    for column in range(
        1,
        codec.tuple_width,
    ):
        mask[:, column, argument_ids] = True

    return mask


# ============================================================
# Transformer denoiser
# ============================================================

class PlanningDFM(nn.Module):
    """
    One Transformer models

        p_theta(
            X_1^i
            |
            X_t,
            initial state,
            goal,
            t
        )

    Context is fixed.

    Only the plan participates in the discrete flow.
    """

    def __init__(
        self,
        config: DFMConfig,
        plan_output_mask: torch.BoolTensor | None = None,
    ):
        super().__init__()

        self.config = config

        self.context_tokens = (
            config.context_rows
            * config.tuple_width
        )

        self.plan_tokens = (
            config.plan_slots
            * config.tuple_width
        )

        self.total_tokens = (
            self.context_tokens
            + self.plan_tokens
        )

        # ----------------------------------------------------
        # Token embedding
        # ----------------------------------------------------

        self.token_embedding = nn.Embedding(
            config.vocab_size,
            config.d_model,
        )

        # ----------------------------------------------------
        # Absolute position
        # ----------------------------------------------------

        self.position_embedding = nn.Embedding(
            self.total_tokens,
            config.d_model,
        )

        # ----------------------------------------------------
        # Segment:
        #
        # 0 = context
        # 1 = plan
        # ----------------------------------------------------

        self.segment_embedding = nn.Embedding(
            2,
            config.d_model,
        )

        # ----------------------------------------------------
        # Transformer
        # ----------------------------------------------------

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.nhead,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu",
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=config.num_layers,
        )

        self.final_norm = nn.LayerNorm(
            config.d_model
        )

        self.output_head = nn.Linear(
            config.d_model,
            config.vocab_size,
        )

        # ----------------------------------------------------
        # Static legal-token mask
        # ----------------------------------------------------

        if plan_output_mask is not None:

            expected = (
                config.plan_slots,
                config.tuple_width,
                config.vocab_size,
            )

            if tuple(
                plan_output_mask.shape
            ) != expected:
                raise ValueError(
                    "Wrong plan output mask shape. "
                    f"Expected {expected}, got "
                    f"{tuple(plan_output_mask.shape)}."
                )

            self.register_buffer(
                "plan_output_mask",
                plan_output_mask.bool(),
            )

        else:
            self.plan_output_mask = None

    def forward(
        self,
        context: torch.LongTensor,
        context_mask: torch.BoolTensor,
        x_t: torch.LongTensor,
        t: torch.Tensor,
    ) -> torch.Tensor:
        """
        Parameters
        ----------
        context:
            [B, context_rows, tuple_width]

        context_mask:
            [B, context_rows]

            True = real context row
            False = padded context row

        x_t:
            [B, plan_slots, tuple_width]

        t:
            [B]

        Returns
        -------
        logits:
            [
                B,
                plan_slots,
                tuple_width,
                vocab_size
            ]
        """

        B = context.shape[0]

        # ----------------------------------------------------
        # Shape checks
        # ----------------------------------------------------

        expected_context = (
            B,
            self.config.context_rows,
            self.config.tuple_width,
        )

        if tuple(context.shape) != expected_context:
            raise ValueError(
                f"Expected context shape "
                f"{expected_context}, got "
                f"{tuple(context.shape)}."
            )

        expected_plan = (
            B,
            self.config.plan_slots,
            self.config.tuple_width,
        )

        if tuple(x_t.shape) != expected_plan:
            raise ValueError(
                f"Expected plan shape "
                f"{expected_plan}, got "
                f"{tuple(x_t.shape)}."
            )

        # ----------------------------------------------------
        # Flatten symbolic tuple structure only at the
        # Transformer boundary.
        # ----------------------------------------------------

        context_flat = context.reshape(
            B,
            -1,
        )

        plan_flat = x_t.reshape(
            B,
            -1,
        )

        tokens = torch.cat(
            [
                context_flat,
                plan_flat,
            ],
            dim=1,
        )

        # ----------------------------------------------------
        # Basic token embeddings
        # ----------------------------------------------------

        h = self.token_embedding(
            tokens
        )

        # ----------------------------------------------------
        # Position embeddings
        # ----------------------------------------------------

        position_ids = torch.arange(
            self.total_tokens,
            device=tokens.device,
        )

        h = (
            h
            + self.position_embedding(
                position_ids
            )[None, :, :]
        )

        # ----------------------------------------------------
        # Context/plan segment embeddings
        # ----------------------------------------------------

        segment_ids = torch.cat(
            [
                torch.zeros(
                    self.context_tokens,
                    dtype=torch.long,
                    device=tokens.device,
                ),
                torch.ones(
                    self.plan_tokens,
                    dtype=torch.long,
                    device=tokens.device,
                ),
            ]
        )

        h = (
            h
            + self.segment_embedding(
                segment_ids
            )[None, :, :]
        )

        # ----------------------------------------------------
        # Add continuous time information only to plan tokens.
        # ----------------------------------------------------

        time_embedding = (
            sinusoidal_time_embedding(
                t,
                self.config.d_model,
            )
        )

        h[
            :,
            self.context_tokens:,
            :
        ] += time_embedding[:, None, :]

        # ----------------------------------------------------
        # Transformer padding mask
        #
        # PyTorch convention:
        #
        # True = IGNORE token
        # ----------------------------------------------------

        context_token_active = (
            context_mask
            .repeat_interleave(
                self.config.tuple_width,
                dim=1,
            )
        )

        plan_token_active = torch.ones(
            (
                B,
                self.plan_tokens,
            ),
            dtype=torch.bool,
            device=tokens.device,
        )

        active = torch.cat(
            [
                context_token_active,
                plan_token_active,
            ],
            dim=1,
        )

        padding_mask = ~active

        # ----------------------------------------------------
        # Transformer
        # ----------------------------------------------------

        h = self.transformer(
            h,
            src_key_padding_mask=padding_mask,
        )

        h = self.final_norm(h)

        # ----------------------------------------------------
        # Keep only plan outputs.
        # ----------------------------------------------------

        plan_h = h[
            :,
            self.context_tokens:,
            :
        ]

        logits = self.output_head(
            plan_h
        )

        logits = logits.reshape(
            B,
            self.config.plan_slots,
            self.config.tuple_width,
            self.config.vocab_size,
        )

        # ----------------------------------------------------
        # Remove impossible token categories.
        # ----------------------------------------------------

        if self.plan_output_mask is not None:

            logits = logits.masked_fill(
                ~self.plan_output_mask[None],
                float("-inf"),
            )

        return logits


# ============================================================
# Probability schedule
# ============================================================

class LinearSchedule:
    """
    kappa_t = t

    dot{kappa}_t = 1

    This is the simplest DFM scheduler.
    """

    @staticmethod
    def kappa(
        t: torch.Tensor,
    ) -> torch.Tensor:
        return t

    @staticmethod
    def dkappa(
        t: torch.Tensor,
    ) -> torch.Tensor:
        return torch.ones_like(t)


# ============================================================
# Forward probability path
# ============================================================

def sample_conditional_path(
    x_1: torch.LongTensor,
    t: torch.Tensor,
    mask_id: int,
    schedule=LinearSchedule,
) -> torch.LongTensor:
    """
    Sample X_t from

        p_t(x^i | x_0, x_1)
        =
        (1-kappa_t) delta_MASK
        +
        kappa_t delta_x1

    independently for each categorical coordinate.

    Parameters
    ----------
    x_1:
        Clean target plan.

        [B, plan_slots, tuple_width]

    t:
        [B]

    Returns
    -------
    x_t:
        Same shape as x_1.
    """

    B = x_1.shape[0]

    kappa = schedule.kappa(t)

    kappa = kappa.reshape(
        B,
        1,
        1,
    )

    reveal_target = (
        torch.rand(
            x_1.shape,
            device=x_1.device,
        )
        < kappa
    )

    x_0 = torch.full_like(
        x_1,
        mask_id,
    )

    x_t = torch.where(
        reveal_target,
        x_1,
        x_0,
    )

    return x_t


# ============================================================
# DFM training loss
# ============================================================

def dfm_loss(
    logits: torch.Tensor,
    x_1: torch.LongTensor,
) -> torch.Tensor:
    """
    Standard categorical denoiser loss.

    The paper's simple probability-denoiser objective is
    cross entropy against X_1 at every coordinate.
    """

    V = logits.shape[-1]

    return F.cross_entropy(
        logits.reshape(-1, V),
        x_1.reshape(-1),
    )


# ============================================================
# Dataset
# ============================================================

class PlanningDFMDataset(Dataset):
    """
    Dataset containing:

        context
        context_mask
        clean target plan X_1

    Negative/non-goal trajectories are intentionally NOT included
    in vanilla DFM training.

    DFM learns the distribution of valid plans.
    """

    def __init__(
        self,
        examples: Sequence[dict],
    ):
        self.examples = list(
            examples
        )

    def __len__(self):
        return len(self.examples)

    def __getitem__(
        self,
        index,
    ):
        example = self.examples[
            index
        ]

        return {
            "context":
                example["context"],

            "context_mask":
                example["context_mask"],

            "plan":
                example["plan"],
        }

    @classmethod
    def from_single_problem(
        cls,
        problem,
        plans,
        codec,
    ):
        """
        Convenience constructor when all plans correspond to the
        same initial state / goal.
        """

        encoded_context = (
            codec.encode_problem_context(
                problem
            )
        )

        examples = []

        for plan in plans:

            encoded_plan = (
                codec.encode_plan(
                    plan
                )
            )

            examples.append(
                {
                    "context":
                        encoded_context.tensor,

                    "context_mask":
                        encoded_context.attention_mask,

                    "plan":
                        encoded_plan.tensor,
                }
            )

        return cls(examples)


# ============================================================
# Training loop
# ============================================================

def train_dfm(
    model: PlanningDFM,
    train_loader: DataLoader,
    codec,
    *,
    epochs: int = 50,
    learning_rate: float = 3e-4,
    weight_decay: float = 1e-4,
    device=None,
    schedule=LinearSchedule,
    grad_clip: float = 1.0,
):
    """
    Train the DFM probability denoiser.
    """

    if device is None:

        device = (
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

    device = torch.device(device)

    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        total_loss = 0.0
        total_examples = 0

        masked_correct = 0
        masked_total = 0

        for batch in train_loader:

            context = (
                batch["context"]
                .to(device)
            )

            context_mask = (
                batch["context_mask"]
                .to(device)
            )

            x_1 = (
                batch["plan"]
                .to(device)
            )

            B = x_1.shape[0]

            # ------------------------------------------------
            # Random flow time
            # ------------------------------------------------

            t = torch.rand(
                B,
                device=device,
            )

            # ------------------------------------------------
            # Sample intermediate plan X_t
            # ------------------------------------------------

            x_t = sample_conditional_path(
                x_1=x_1,
                t=t,
                mask_id=codec.mask_id,
                schedule=schedule,
            )

            # ------------------------------------------------
            # Predict clean target posterior
            # ------------------------------------------------

            logits = model(
                context=context,
                context_mask=context_mask,
                x_t=x_t,
                t=t,
            )

            loss = dfm_loss(
                logits,
                x_1,
            )

            # ------------------------------------------------
            # Optimizer
            # ------------------------------------------------

            optimizer.zero_grad(
                set_to_none=True
            )

            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    grad_clip,
                )

            optimizer.step()

            # ------------------------------------------------
            # Statistics
            # ------------------------------------------------

            total_loss += (
                loss.item()
                * B
            )

            total_examples += B

            with torch.no_grad():

                prediction = (
                    logits.argmax(
                        dim=-1
                    )
                )

                # Accuracy only where X_t was still MASK.
                # This is more informative than counting tokens
                # already revealed to the model.
                mask_positions = (
                    x_t
                    != x_1
                )

                if mask_positions.any():

                    masked_correct += (
                        prediction[
                            mask_positions
                        ]
                        ==
                        x_1[
                            mask_positions
                        ]
                    ).sum().item()

                    masked_total += (
                        mask_positions
                        .sum()
                        .item()
                    )

        average_loss = (
            total_loss
            / max(
                total_examples,
                1,
            )
        )

        masked_accuracy = (
            masked_correct
            / max(
                masked_total,
                1,
            )
        )

        history.append(
            {
                "epoch": epoch,
                "loss": average_loss,
                "masked_accuracy":
                    masked_accuracy,
            }
        )

        print(
            f"Epoch {epoch:03d} | "
            f"loss={average_loss:.4f} | "
            f"masked accuracy="
            f"{masked_accuracy:.4f}"
        )

    return history


# ============================================================
# Euler / CTMC DFM sampling
# ============================================================

@torch.no_grad()
def sample_dfm(
    model: PlanningDFM,
    context: torch.LongTensor,
    context_mask: torch.BoolTensor,
    codec,
    *,
    num_steps: int = 128,
    schedule=LinearSchedule,
    temperature: float = 1.0,
    device: str | torch.device | None = None,
):
    """
    Generate a plan using the DFM probability velocity.

    Starts from:

        X_0 = all MASK

    and repeatedly applies

        delta_Xt
        +
        h * u_t

    where

        u_t =
        dot{kappa}/(1-kappa)
        (p_theta - delta_Xt).

    Equivalently:

        transition distribution
        =
        (1-alpha) delta_Xt
        +
        alpha p_theta

    with

        alpha =
        h * dot{kappa}/(1-kappa).
    """

    if num_steps <= 0:
        raise ValueError(
            "num_steps must be positive."
        )

    if device is None:

        device = next(
            model.parameters()
        ).device

    device = torch.device(device)

    model.eval()

    # --------------------------------------------------------
    # Add batch dimension if necessary.
    # --------------------------------------------------------

    single_sample = (
        context.ndim == 2
    )

    if single_sample:

        context = context.unsqueeze(0)

        context_mask = (
            context_mask
            .unsqueeze(0)
        )

    context = context.to(device)

    context_mask = (
        context_mask.to(device)
    )

    B = context.shape[0]

    # --------------------------------------------------------
    # X_0 = fully masked plan
    # --------------------------------------------------------

    x = torch.full(
        (
            B,
            codec.num_plan_slots,
            codec.tuple_width,
        ),
        codec.mask_id,
        dtype=torch.long,
        device=device,
    )

    h = 1.0 / num_steps

    # --------------------------------------------------------
    # Euler integration
    # --------------------------------------------------------

    for step in range(
        num_steps
    ):

        t_value = (
            step
            / num_steps
        )

        t = torch.full(
            (B,),
            t_value,
            device=device,
        )

        logits = model(
            context=context,
            context_mask=context_mask,
            x_t=x,
            t=t,
        )

        if temperature != 1.0:

            logits = (
                logits
                / temperature
            )

        posterior = F.softmax(
            logits,
            dim=-1,
        )

        kappa = schedule.kappa(t)

        dkappa = schedule.dkappa(t)

        rate = (
            dkappa
            /
            (1.0 - kappa)
        )

        alpha = h * rate

        # ----------------------------------------------------
        # For Euler sampling to define a valid probability
        # distribution, alpha must not exceed one.
        #
        # Linear schedule + uniform steps automatically obeys
        # this.
        # ----------------------------------------------------

        if (
            alpha.max().item()
            > 1.0 + 1e-6
        ):
            raise ValueError(
                "Sampling step is too large for "
                "this scheduler: "
                f"max alpha={alpha.max().item():.4f}. "
                "Increase num_steps."
            )

        alpha = alpha.reshape(
            B,
            1,
            1,
            1,
        )

        # ----------------------------------------------------
        # q =
        #   (1-alpha) delta_current
        #   +
        #   alpha * p_theta
        # ----------------------------------------------------

        transition = (
            alpha
            * posterior
        )

        stay_probability = (
            1.0 - alpha
        )

        transition.scatter_add_(
            dim=-1,
            index=x.unsqueeze(-1),
            src=stay_probability.expand(
                B,
                codec.num_plan_slots,
                codec.tuple_width,
                1,
            ),
        )

        # Numerical safety.
        transition = torch.clamp(
            transition,
            min=0.0,
        )

        transition = (
            transition
            /
            transition.sum(
                dim=-1,
                keepdim=True,
            )
        )

        # ----------------------------------------------------
        # Independently sample each categorical coordinate.
        # ----------------------------------------------------

        flat_distribution = (
            transition.reshape(
                -1,
                codec.vocab_size,
            )
        )

        sampled = torch.multinomial(
            flat_distribution,
            num_samples=1,
        )

        x = sampled.reshape(
            B,
            codec.num_plan_slots,
            codec.tuple_width,
        )

    if single_sample:
        x = x.squeeze(0)

    return x